In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
from matplotlib import pyplot as plt
import seaborn as sns

load_dotenv("../.env")
load_dotenv("../.env.local")




In [ ]:
# connect to your mart
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

with engine.connect() as conn:
    print("Connected successfully!")
    print(conn.execute(text("SELECT current_database();")).scalar())

query = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_name = 'fct_ed_visits';
"""

with engine.connect() as conn:
    print(conn.execute(text(query)).fetchall())

In [ ]:
# Analysis 1 — admission rate by chief complaint category
query = """
SELECT
    cc.complaint_category,
    COUNT(*) AS total_visits,
    SUM(CASE WHEN e.disposition = 'admitted' THEN 1 ELSE 0 END) AS admissions,
    ROUND(100.0 * SUM(CASE WHEN e.disposition = 'admitted' THEN 1 ELSE 0 END) / COUNT(*), 1) AS admission_rate_pct
FROM dbt_mrt.fct_ed_visits e
JOIN dbt_mrt.bridge_triage_complaints b ON e.stay_id = b.stay_id
JOIN dbt_mrt.dim_chiefcomplaint cc ON b.complaint_id = cc.complaint_id
GROUP BY cc.complaint_category
ORDER BY admission_rate_pct DESC
"""

# Analysis 1 — bar chart of admission rates by complaint category
df1 = pd.read_sql_query(text(query), engine)

plt.figure(figsize=(12, 6))
sns.barplot(data=df1, x='complaint_category', y='admission_rate_pct', palette='Blues_d',hue='complaint_category',legend=False)
plt.title('Admission Rate by Chief Complaint Category')
plt.xlabel('Complaint Category')
plt.ylabel('Admission Rate (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../docs/schema/admission_by_complaint.png')
plt.show()

In [ ]:
# Analysis 2 — average length of stay by acuity level
query2 = """
SELECT
    acuity_level,
    COUNT(*) AS visits,
    ROUND(
        AVG(
            (discharge_date - arrival_date) * 24
            + (
                CASE
                    WHEN (td.hour_num * 60 + td.minute_num) >= (ta.hour_num * 60 + ta.minute_num)
                    THEN (
                        (td.hour_num * 60 + td.minute_num)
                        -
                        (ta.hour_num * 60 + ta.minute_num)
                    ) / 60.0
                    ELSE (
                        (1440 - (ta.hour_num * 60 + ta.minute_num))
                        +
                        (td.hour_num * 60 + td.minute_num)
                    ) / 60.0
                END
            )
        ),
        2
    ) AS avg_los_hours
FROM dbt_mrt.fct_ed_visits e
join dbt_mrt.dim_hour td on e.discharge_hour = td.time_hhmm
join dbt_mrt.dim_hour ta on e.arrival_hour = ta.time_hhmm
WHERE acuity_level IS NOT NULL
GROUP BY acuity_level
ORDER BY acuity_level
"""

df2 = pd.read_sql_query(text(query2), engine)
plt.figure(figsize=(12, 6))
sns.barplot(data=df2, x='acuity_level', y='avg_los_hours', palette='Blues_d' ,hue='acuity_level',legend=False)
plt.title('Acuity average staying hours')
plt.xlabel('acuity level')
plt.ylabel('Admission hours')
plt.xticks( ha='right')
plt.tight_layout()
plt.savefig('../docs/schema/avg_hours_by_acuity.png')
plt.show()

In [ ]:
# Analysis 3 — ED visit volume by time of day
query3 = """
SELECT
    h.time_of_day,
    h.shift_type,
    COUNT(*) AS visit_count
FROM dbt_mrt.fct_ed_visits e
JOIN dbt_mrt.dim_hour h ON e.arrival_hour = h.time_hhmm
GROUP BY h.time_of_day, h.shift_type
ORDER BY visit_count DESC
"""

df3 = pd.read_sql_query(text(query3), engine)
plt.figure(figsize=(12, 6))
sns.barplot(data=df3, x='time_of_day', y='visit_count', palette={'Business-Hours': '#2196F3', 'Off-Hours': '#FF9800'},
    order=['Night', 'Morning', 'Afternoon', 'Evening'] , hue="shift_type" ,legend=True)
plt.title('ED Visit Volume by Time of Day and Shift Type')
plt.xlabel('Time of Day')
plt.ylabel('Number of Visits')
plt.tight_layout()
plt.savefig('../docs/schema/visits_by_timeofday.png')
plt.show()

In [ ]:
# Analysis 4 — top 10 most common diagnoses
query4 = """
SELECT
    i.icd_title,
    i.icd_version,

    COUNT(*) AS frequency,
    SUM(CASE WHEN e.disposition = 'admitted' THEN 1 ELSE 0 END) AS admissions
FROM dbt_mrt.fct_diagnosis d
JOIN dbt_mrt.dim_icd_classification i ON d.icd_id = i.icd_id
JOIN dbt_mrt.fct_ed_visits e ON d.stay_id = e.stay_id
WHERE d.diagnosis_order = 1
GROUP BY i.icd_title, i.icd_version,i.icd_code
ORDER BY frequency DESC
LIMIT 10
"""

df4 = pd.read_sql_query(text(query4), engine)
df4

In [ ]:
# Analysis 5 — acuity distribution by pain level
query5 = """
SELECT
	pain_level,
    acuity_level,
    COUNT(*) AS visits
FROM dbt_mrt.fct_ed_visits
WHERE acuity_level IS NOT NULL
AND pain_level IS NOT NULL
GROUP BY pain_level, acuity_level
ORDER BY pain_level,acuity_level
"""

df5 = pd.read_sql_query(text(query5), engine)
plt.figure(figsize=(10, 6))
sns.barplot(data=df5, x='pain_level', y='visits', palette={1: '#ff0000', 2: '#ff8503', 3: '#fce112', 4: '#1fa80a'},
    order=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9','10'] , hue="acuity_level" ,legend=True)
plt.title('ED Visit Volume by pain score and acuity')
plt.xlabel('Pain score')
plt.ylabel('Number of Visits')
plt.tight_layout()
plt.savefig('../docs/schema/visits_by_pain.png')
plt.show()


In [ ]:
# Analysis 6 — top 120 medications frequency by season
query6 = """
select m.medication_name,
	d.date_season,
	count(p.med_id) as frequency
from dbt_mrt.dim_medications m
join dbt_mrt.fct_pyxis p
	on m.med_id = p.med_id
join dbt_mrt.dim_date d
	on p.dispensing_date = d.full_date
group by m.medication_name,d.date_season
order by frequency desc
limit 120;
"""

df6 = pd.read_sql_query(text(query6), engine)
pivot = df6.pivot(
    index="medication_name",
    columns="date_season",
    values="frequency"
).fillna(0)

plt.figure(figsize=(17,20))
plt.imshow(pivot)

plt.xticks(range(len(pivot.columns)), pivot.columns,rotation=70)
plt.yticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)

plt.colorbar(label="Frequency")
plt.tight_layout()
plt.savefig('../docs/schema/visits_by_pain.png')
plt.show()

In [ ]:
# Analysis 7 — Medication Class Usage by Diagnosis
query7 = """
SELECT
    e.etc_description AS drug_class,
    i.icd_title AS diagnosis,
    i.icd_version,
    COUNT(DISTINCT d.stay_id) AS visits_count,
    COUNT(p.med_id) AS medication_events,
    COUNT(DISTINCT p.med_id) AS unique_medications
FROM dbt_mrt.fct_diagnosis d
JOIN dbt_mrt.fct_pyxis p
    ON p.stay_id = d.stay_id
JOIN dbt_mrt.dim_medications m
    ON m.med_id = p.med_id
JOIN dbt_mrt.dim_etc_classification e
    ON e.etc_code = m.etc_code
JOIN dbt_mrt.dim_icd_classification i
    ON i.icd_id = d.icd_id
GROUP BY
    e.etc_description,
    i.icd_title,
    i.icd_version
ORDER BY
    visits_count DESC;
"""

df7 = pd.read_sql_query(text(query7), engine)
pivot = df7.pivot_table(
    index="drug_class",
    columns="diagnosis",
    values="visits_count",
    aggfunc="sum",
    fill_value=0
)

plt.figure(figsize=(15,8))
sns.heatmap(pivot, cmap="viridis")

plt.title("Medication Class Usage by Diagnosis")
plt.xlabel("Diagnosis")
plt.ylabel("Drug Class")
plt.savefig('../docs/schema/med_usage_by_diagnosis.png')
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

In [ ]:
# load features from mart
ml_query = """
SELECT
    e.acuity_level,
    e.pain_level,
    e.heart_rate,
    e.o2_saturation,
    e.systolic_bp,
    e.diastolic_bp,
    e.temperature,
    e.resp_rate,
    e.disposition
FROM dbt_mrt.fct_ed_visits e
WHERE acuity_level IS NOT NULL
AND disposition IS NOT NULL
"""

df_ml = pd.read_sql_query(text(ml_query), engine)

In [ ]:
# prepare features
features = ['acuity_level', 'pain_level', 'heart_rate',
            'o2_saturation', 'systolic_bp', 'diastolic_bp',
            'temperature', 'resp_rate']

X = df_ml[features].fillna(df_ml[features].median())
y = df_ml['disposition'].apply(lambda x : 1 if x=='admitted' else 0).astype(int)

# split and train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# evaluate
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

# feature importance
importance = pd.DataFrame({
    'feature': features,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)
print(importance)